# fp2mp_core Artifacts Demo
Runs a question and displays the extended run log plus generated/found images.

In [1]:
MODEL = 'gpt-4o-mini'
MAX_ITERATIONS = 5
QUESTION = 'Создай мастер-план города Гатчина'

In [ ]:
from fp2mp_core.graph import run
result = run(input=QUESTION, model=MODEL, max_iterations=MAX_ITERATIONS)
result['artifacts']

c:\Users\Eynor\AppData\Local\Programs\Python\Python311\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-26 11:19:21.995 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-05-26 11:19:21.999 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-05-26 11:19:22.008 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-05-26 11:19:22.013 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving 

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result['output']))

In [ ]:
from pathlib import Path
run_dir = Path(result['artifacts']['run_dir'])
display(Markdown((run_dir / 'run_log.md').read_text(encoding='utf-8')))

In [ ]:
import json
import pandas as pd
log = json.loads((run_dir / 'run_log.json').read_text(encoding='utf-8'))
calls = pd.DataFrame(log.get('tool_calls', []))
if not calls.empty:
    calls['observation'] = calls['observation'].astype(str).str.slice(0, 240)
calls

In [ ]:
from IPython.display import Image
groups = [('Карты BlocksNet', 'map'), ('Визуализации CodeSpatial', 'code_image'), ('Найдено в вебе', 'web_image')]
for title, kind in groups:
    files = [f for f in log.get('saved_files', []) if f.get('kind') == kind]
    display(Markdown(f'## {title}'))
    if not files:
        display(Markdown('_Нет файлов_'))
        continue
    for f in files:
        path = f['path']
        caption = f'`{Path(path).name}` · {kind} · {f.get("source", f.get("meta", {}).get("page", ""))}'
        display(Markdown(caption))
        display(Image(filename=path))

In [ ]:
data_files = pd.DataFrame([f for f in log.get('saved_files', []) if f.get('kind') == 'csv'])
data_files[['path', 'size']] if not data_files.empty else data_files

In [ ]:
agent_outputs = log.get('agent_outputs', [])
blocksnet_outputs = [o for o in agent_outputs if o.get('type') == 'blocksnet_result']
display(Markdown('## Выводы агентов: BlocksNet'))
if not blocksnet_outputs:
    display(Markdown('_Нет blocksnet_result в agent_outputs_'))
for output in blocksnet_outputs:
    header = (
        f"### {output.get('agent')} / {output.get('sub_query_id', '')} "
        f"conf={output.get('confidence')} / {output.get('promotion', 'unknown')}"
    )
    display(Markdown(header))
    display(Markdown('```text\n' + output.get('content', '') + '\n```'))

pd.DataFrame(agent_outputs)[['agent', 'type', 'sub_query_id', 'confidence', 'promotion']] if agent_outputs else pd.DataFrame()